In [9]:
import pandas as pd
import folium

# 1. Загружаем данные точек
df_points = pd.read_csv('csv_файлов_по_районам/map_1_points_data.csv')

# Вытаскиваем транслит района из имени файла (до первого подчеркивания)
df_points['district_translit'] = df_points['source_file'].apply(lambda x: x.split('_')[0])

# Словарь-переводчик для ЮЗАО (транслит из ваших файлов -> русский язык для GeoJSON)
id_to_russian = {
    'Gagarinskiy': 'Гагаринский',
    'Lomonosovskiy': 'Ломоносовский',
    'Akademicheskaya': 'Академический', # Проверьте, как именно в GeoJSON (Академический или Академическая)
    'Kotlovka': 'Котловка',
    'Zuzino': 'Зюзино',
    'Cheremushki': 'Черемушки',
    'Obruchevskiy': 'Обручевский',
    'Konkovo': 'Коньково',
    'Tepliy': 'Теплый стан',       # Обычно в геоданных пишется «Теплый Стан»
    'Yasenevo': 'Ясенево',
    'NButovo': 'Северное Бутово',  # Проверьте точное написание в вашем GeoJSON
    'SButovo': 'Южное Бутово'      # Проверьте точное написание в вашем GeoJSON
}

# Переводим названия районов на русский язык
df_points['district_name'] = df_points['district_translit'].map(id_to_russian)

# Группируем по русскому названию и считаем среднее для всего района
district_averages = df_points.groupby('district_name')['PM2.5'].mean().reset_index()

# 2. Создаем базовую карту
m = folium.Map(location=[55.6500, 37.5500], zoom_start=11, tiles='CartoDB positron')

# 3. Строим хороплет
folium.Choropleth(
    geo_data="moscow_districts.geojson",        # Ваш файл с границами
    name="Хороплет PM2.5",
    data=district_averages,
    columns=['district_name', 'PM2.5'],
    key_on='feature.properties.district',               # Имя свойства в GeoJSON (NAME или name - проверьте регистр!)
    fill_color='RdYlGn_r',                             # Желтый -> Оранжевый -> Красный
    fill_opacity=0.6,
    line_opacity=0.3,
    legend_name='Средняя концентрация PM2.5 (мкг/м³)',
    highlight=True
).add_to(m)

# 4. Накладываем сверху ваши исходные точки-капли из прошлого шага (со скорректированными порогами!)
for index, row in df_points.iterrows():
    if pd.isna(row['latitude']) or pd.isna(row['longitude']):
        continue
        
    if row['PM2.5'] <= 15:
        hex_color = '#2ecc71'
    elif row['PM2.5'] <= 20:
        hex_color = '#e67e22'
    else:
        hex_color = '#e74c3c'
        
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        tooltip=f"{row['source_file']}: {row['PM2.5']:.1f} мкг/м³",
        icon=folium.DivIcon(
            icon_size=(16, 22),
            icon_anchor=(8, 22),
            html=f"""
            <div style="position: relative; width: 16px; height: 22px;">
                <div style="position: absolute; width: 16px; height: 16px; background-color: {hex_color}; border-radius: 50% 50% 50% 0; transform: rotate(-45deg); box-shadow: -1px 2px 4px rgba(0,0,0,0.3);"></div>
                <div style="position: absolute; top: 5px; left: 5px; width: 6px; height: 6px; background-color: #ffffff; border-radius: 50%;"></div>
            </div>
            """
        )
    ).add_to(m)

m.save('yzao_choropleth_map.html')
print("🎉 Хороплет успешно собран!")

🎉 Хороплет успешно собран!


In [10]:
# 1. Сортируем районы по возрастанию PM2.5 (от чистых к грязным)
# Если хотите от грязных к чистым, поставьте ascending=False
rating_list = district_averages.sort_values(by='PM2.5', ascending=True).reset_index(drop=True)

# Добавляем колонку с местом в рейтинге (начиная с 1)
rating_list.index = rating_list.index + 1
rating_list = rating_list.reset_index().rename(columns={'index': 'Место'})

# 2. Красиво выводим рейтинг в терминал для проверки
print("\n🏆 ЭКОЛОГИЧЕСКИЙ РЕЙТИНГ РАЙОНОВ ЮЗАО (по уровню PM2.5):")
print("=" * 55)
for index, row in rating_list.iterrows():
    print(f"{row['Место']}. Район {row['district_name']:<20} | Среднее PM2.5: {row['PM2.5']:.2f} мкг/м³")
print("=" * 55)

# 3. Автоматически сохраняем рейтинг в Excel для создания красивой таблицы на Слайд 8
rating_list.to_excel('yzao_ecology_rating.xlsx', index=False)
print("📊 Рейтинг сохранен в файл 'yzao_ecology_rating.xlsx'!")


🏆 ЭКОЛОГИЧЕСКИЙ РЕЙТИНГ РАЙОНОВ ЮЗАО (по уровню PM2.5):
1. Район Гагаринский          | Среднее PM2.5: 14.19 мкг/м³
2. Район Ломоносовский        | Среднее PM2.5: 14.45 мкг/м³
3. Район Зюзино               | Среднее PM2.5: 15.31 мкг/м³
4. Район Черемушки            | Среднее PM2.5: 15.97 мкг/м³
5. Район Академический        | Среднее PM2.5: 16.09 мкг/м³
6. Район Котловка             | Среднее PM2.5: 16.86 мкг/м³
7. Район Обручевский          | Среднее PM2.5: 17.57 мкг/м³
8. Район Коньково             | Среднее PM2.5: 19.12 мкг/м³
9. Район Теплый стан          | Среднее PM2.5: 20.38 мкг/м³
10. Район Ясенево              | Среднее PM2.5: 21.60 мкг/м³
11. Район Северное Бутово      | Среднее PM2.5: 24.48 мкг/м³
12. Район Южное Бутово         | Среднее PM2.5: 28.04 мкг/м³


ModuleNotFoundError: No module named 'openpyxl'